# 03 — Extended Isolation Forest (EIF)

## Contexto

Tercer notebook de la serie. Isolation Forest (notebook 02) particiona el espacio de features con
cortes paralelos a los ejes (elige una feature y un umbral al azar en cada nodo). Cuando la
distribución de los datos normales no está alineada con los ejes, esos cortes producen un sesgo
sistemático en el mapa de anomalía (artefactos en forma de rejilla/cruz alrededor del origen),
descrito por **Hariri, Kind & Brunner (2019), "Extended Isolation Forest"**. EIF generaliza el
corte de cada nodo a un **hiperplano con orientación aleatoria** (vector normal aleatorio en vez
de un eje canónico), controlado por el hiperparámetro `extension_level` (0 = equivalente a IF
clásico; `n_features-1` = máxima generalización). Este es, además, el modelo que ya se había
explorado de forma exploratoria en este repositorio (`FFT_train_eif_v2.ipynb`,
`FFT_train_hibrido.ipynb`) — aquí se reimplementa sobre `utils/` para que el pipeline de
extracción de características, evaluación y guardado de resultados sea idéntico al de los
notebooks 02 y 04, y la comparación del notebook 05 sea homogénea.

## Qué se hace

Igual que en el notebook 02: se entrena y evalúa EIF (vía H2O) sobre las tres variantes de fuente
de señal (eléctrica, vibración, híbrida), solo con datos sanos, con Optuna ajustando
hiperparámetros para maximizar la compacidad de la distribución de scores de los sanos, y umbral
como percentil de esa distribución.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import h2o
from h2o.estimators import H2OExtendedIsolationForestEstimator

from utils.config import (
    fijar_semillas, SEED, RUTA_FEATURES, RUTA_RESULTADOS,
    NOMBRES_COLS_ELEC, NOMBRES_COLS_VIB, NOMBRES_COLS, VENTANAS_POR_EXP,
)
from utils.data import cargar_indice
from utils.eval import agregar_por_experimento, agregar_variable, calcular_metricas, guardar_resultado

fijar_semillas()
optuna.logging.set_verbosity(optuna.logging.WARNING)

COLS_POR_FUENTE = {
    "electrica": NOMBRES_COLS_ELEC,
    "vibracion": NOMBRES_COLS_VIB,
    "hibrida":   NOMBRES_COLS,
}

FALLOS_BARRA_ROTA = {"e", "b", "v", "p"}
FALLOS_RODAMIENTO = {"g", "o", "r", "c"}

def familia(maquina):
    if maquina == "h":
        return "sano"
    if maquina in FALLOS_BARRA_ROTA:
        return "barra_rota"
    if maquina in FALLOS_RODAMIENTO:
        return "rodamiento"
    return "otro"

index = cargar_indice()


In [ ]:
h2o.init(nthreads=-1, max_mem_size="6G")


## Carga de datos (una sola vez, se reutiliza para las 3 variantes)

Se importan los CSV completos como `H2OFrame` una única vez; cada variante de fuente solo
selecciona el subconjunto de columnas correspondiente (`u_bin*`, `v_bin*`, `w_bin*` para
eléctrica; el resto para vibración; todas para híbrida).


In [ ]:
train_full = h2o.import_file(os.path.join(RUTA_FEATURES, "train", "sano_train.csv"))
val_full   = h2o.import_file(os.path.join(RUTA_FEATURES, "val", "sano.csv"))

carpeta_test = os.path.join(RUTA_FEATURES, "test")
archivos_test = sorted(f for f in os.listdir(carpeta_test) if f.endswith(".csv"))
test_frames = {a: h2o.import_file(os.path.join(carpeta_test, a)) for a in archivos_test}
print(f"{len(archivos_test)} grupos de test cargados")


## Función de entrenamiento + evaluación (reutilizable para las 3 variantes)

Mismo criterio que el notebook 02: Optuna minimiza `std/rango` de las medianas de score de los
experimentos sanos de train. `extension_level` se busca entre 0 (equivale a IF clásico) y
`min(50, n_features-1)`: en H2O, el coste de construir cada árbol de EIF crece muy rápido con
`extension_level` en dimensión alta (con 5608 features, permitir `extension_level` cercano al
máximo hace que un solo trial de Optuna tarde más de 30 minutos — se probó y se abortó por
timeout). El propio pipeline EIF exploratorio ya existente en este repositorio
(`FFT_train_eif_v2.ipynb`) capa igualmente `extension_level` a 50 por la misma razón práctica, así
que se reutiliza ese límite en vez de `n_features-1`.


In [ ]:
def pipeline_eif(fuente, n_trials=8):
    cols = COLS_POR_FUENTE[fuente]
    ext_max = min(50, len(cols) - 1)
    print(f"\n{'='*60}\nFUENTE: {fuente}  ({len(cols)} features, extension_level<={ext_max})\n{'='*60}")

    train = train_full[cols]
    val   = val_full[cols]

    def objective(trial):
        ntrees          = trial.suggest_int("ntrees", 30, 150)
        sample_size     = trial.suggest_int("sample_size", 32, 128, step=32)
        extension_level = trial.suggest_int("extension_level", 0, ext_max)
        percentil       = trial.suggest_int("percentil_umbral", 90, 99)

        modelo = H2OExtendedIsolationForestEstimator(
            ntrees=ntrees, sample_size=sample_size,
            extension_level=extension_level, seed=SEED,
        )
        modelo.train(training_frame=train)

        scores_train = modelo.predict(train)["anomaly_score"].as_data_frame().values.flatten()
        medianas = agregar_por_experimento(scores_train, VENTANAS_POR_EXP)
        umbral = np.percentile(medianas, percentil)
        rango = medianas.max() - medianas.min()
        compacidad = medianas.std() / rango if rango > 0 else medianas.std()

        trial.set_user_attr("umbral", float(umbral))
        trial.set_user_attr("percentil_umbral", percentil)
        return compacidad

    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(
        direction="minimize", sampler=sampler,
        storage=f"sqlite:///{RUTA_RESULTADOS}/optuna_eif_{fuente}.db",
        study_name=f"eif_{fuente}", load_if_exists=True,
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_trial
    umbral = best.user_attrs["umbral"]
    print(f"Mejores hiperparametros: {best.params}")
    print(f"Umbral (percentil {best.user_attrs['percentil_umbral']} de sanos train): {umbral:.4f}")

    modelo = H2OExtendedIsolationForestEstimator(
        ntrees=best.params["ntrees"], sample_size=best.params["sample_size"],
        extension_level=best.params["extension_level"], seed=SEED,
    )
    modelo.train(training_frame=train)

    def scores_de(frame):
        return modelo.predict(frame)["anomaly_score"].as_data_frame().values.flatten()

    filas = []
    for m in agregar_por_experimento(scores_de(train), VENTANAS_POR_EXP):
        filas.append({"grupo": "sano_train", "maquina": "h", "familia": "sano", "score": m})
    for m in agregar_por_experimento(scores_de(val), VENTANAS_POR_EXP):
        filas.append({"grupo": "sano_val", "maquina": "h", "familia": "sano", "score": m})

    for archivo_csv, frame in test_frames.items():
        nombre_fallo = archivo_csv.replace(".csv", "")
        grupo_idx = index[(index["Split"] == "test") & (index["Fallo"] == nombre_fallo)]
        n_archivos = len(grupo_idx)
        if n_archivos == 0:
            continue
        maquina = grupo_idx["Maquina"].iloc[0]

        scores = scores_de(frame[cols])
        for m in agregar_variable(scores, n_archivos):
            filas.append({
                "grupo": nombre_fallo, "maquina": maquina,
                "familia": familia(maquina), "score": m,
            })

    df_scores = pd.DataFrame(filas)

    metricas_por_familia = {}
    for fam in ["barra_rota", "rodamiento"]:
        subset = df_scores[df_scores["familia"].isin(["sano", fam])]
        y_true = (subset["familia"] == fam).astype(int).values
        metricas_por_familia[fam] = calcular_metricas(y_true, subset["score"].values, umbral)
        print(f"  {fam:12s} -> precision={metricas_por_familia[fam]['precision']:.2f} "
              f"recall={metricas_por_familia[fam]['recall']:.2f} "
              f"f1={metricas_por_familia[fam]['f1']:.2f} auc={metricas_por_familia[fam]['auc']:.3f}")

    y_true_global = (df_scores["familia"] != "sano").astype(int).values
    metricas_global = calcular_metricas(y_true_global, df_scores["score"].values, umbral)
    print(f"  {'global':12s} -> precision={metricas_global['precision']:.2f} "
          f"recall={metricas_global['recall']:.2f} f1={metricas_global['f1']:.2f} "
          f"auc={metricas_global['auc']:.3f}")

    fig, ax = plt.subplots(figsize=(14, 6))
    orden = ["sano_train", "sano_val"] + sorted(
        g for g in df_scores["grupo"].unique() if g not in ("sano_train", "sano_val")
    )
    df_scores.boxplot(column="score", by="grupo", ax=ax, positions=range(len(orden)))
    ax.axhline(umbral, color="red", linestyle="--", label=f"umbral={umbral:.3f}")
    ax.set_xticklabels(orden, rotation=60, ha="right", fontsize=7)
    ax.set_title(f"Extended Isolation Forest — {fuente} — score por grupo")
    plt.suptitle("")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"resultados/03_boxplot_eif_{fuente}.png", dpi=120)
    plt.show()

    resultado = {
        "modelo": "extended_isolation_forest",
        "fuente_senal": fuente,
        "n_features": len(cols),
        "mejores_hiperparametros": best.params,
        "umbral": float(umbral),
        "metricas_por_familia": metricas_por_familia,
        "metricas_global": metricas_global,
    }
    path = guardar_resultado(resultado, f"eif_{fuente}.json")
    print(f"Guardado: {path}")
    return resultado


## Ejecución sobre las 3 variantes de fuente de señal

In [ ]:
resultados_eif = {}
for fuente in ["electrica", "vibracion", "hibrida"]:
    resultados_eif[fuente] = pipeline_eif(fuente, n_trials=8)


## Tabla resumen

In [ ]:
resumen = pd.DataFrame([
    {
        "fuente": fuente,
        "auc_barra_rota": r["metricas_por_familia"]["barra_rota"]["auc"],
        "auc_rodamiento": r["metricas_por_familia"]["rodamiento"]["auc"],
        "f1_global": r["metricas_global"]["f1"],
        "auc_global": r["metricas_global"]["auc"],
    }
    for fuente, r in resultados_eif.items()
])
resumen


In [ ]:
h2o.cluster().shutdown(prompt=False)


## Conclusiones parciales

- La tabla anterior es directamente comparable con la del notebook 02 (mismas familias de fallo,
  mismo criterio de agregación y de umbral): permite ver si generalizar los cortes a hiperplanos
  con orientación aleatoria (EIF) mejora sobre los cortes paralelos a los ejes (IF) en este
  problema concreto, y para qué variante de fuente de señal.
- Ambos modelos comparten la misma ventaja frente al baseline del notebook 01: no requieren ningún
  supuesto sobre el mecanismo físico del fallo ni sobre el régimen de control de la máquina.
- El notebook 04 explora un enfoque completamente distinto (Transformers) sobre el mismo pipeline
  de features y evaluación, para completar la comparación del notebook 05.
